In [0]:
from pyspark.sql import functions as F

df = spark.read.parquet(
    '/Volumes/blueprint_spark/bronze/bronze/vendas.parquet',
    mergeSchema=True,
    )

display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df.groupBy('produto').count().orderBy(F.col("count").desc()).show()

+--------+-----+
| produto|count|
+--------+-----+
|Notebook|    5|
|   Mouse|    3|
| Monitor|    3|
|  Webcam|    2|
| Teclado|    2|
+--------+-----+



In [0]:
df.groupBy('produto',"vendedor").count().orderBy(F.col("count").desc()).show()

+--------+--------+-----+
| produto|vendedor|count|
+--------+--------+-----+
| Teclado|  Elaine|    1|
| Monitor|   Diego|    1|
|Notebook|     Ana|    1|
|Notebook|   Bruno|    1|
|Notebook|   Diego|    1|
|Notebook|  Elaine|    1|
|   Mouse|   Carla|    1|
|  Webcam|   Diego|    1|
| Teclado|   Bruno|    1|
|  Webcam|   Carla|    1|
| Monitor|   Bruno|    1|
|   Mouse|  Elaine|    1|
|   Mouse|     Ana|    1|
|Notebook|   Carla|    1|
| Monitor|     Ana|    1|
+--------+--------+-----+



In [0]:
df.groupBy("produto","regiao").avg("valor").show()

+--------+------------+----------+
| produto|      regiao|avg(valor)|
+--------+------------+----------+
|   Mouse|     Sudeste|      80.0|
| Teclado|         Sul|     220.0|
|Notebook|    Nordeste|    3400.0|
| Monitor|         Sul|     950.0|
|Notebook|       Norte|    3250.0|
| Monitor|Centro-Oeste|     980.0|
| Monitor|     Sudeste|     970.0|
|   Mouse|    Nordeste|      82.0|
|Notebook|     Sudeste|    3200.0|
| Teclado|       Norte|     210.0|
|  Webcam|Centro-Oeste|     145.0|
|  Webcam|    Nordeste|     150.0|
|Notebook|Centro-Oeste|    3100.0|
|Notebook|         Sul|    3300.0|
|   Mouse|       Norte|      75.0|
+--------+------------+----------+



In [0]:
df.groupBy("produto").agg(
            F.count("*").alias("total"),
            F.sum("valor").alias("total_vendas"),
            F.avg("valor").alias("ticket_medio"),
            F.max("valor").alias("max_venda"),
            F.min("valor").alias("min_venda")
).show()

+--------+-----+------------+-----------------+---------+---------+
| produto|total|total_vendas|     ticket_medio|max_venda|min_venda|
+--------+-----+------------+-----------------+---------+---------+
|  Webcam|    2|       295.0|            147.5|    150.0|    145.0|
|   Mouse|    3|       237.0|             79.0|     82.0|     75.0|
| Monitor|    3|      2900.0|966.6666666666666|    980.0|    950.0|
| Teclado|    2|       430.0|            215.0|    220.0|    210.0|
|Notebook|    5|     16250.0|           3250.0|   3400.0|   3100.0|
+--------+-----+------------+-----------------+---------+---------+



In [0]:
df.groupBy("produto","vendedor").agg(
            F.count("*").alias("total"),
            F.sum("valor").alias("total_vendas"),
            F.avg("valor").alias("ticket_medio"),
            F.max("valor").alias("max_venda"),
            F.min("valor").alias("min_venda")
).orderBy(F.col("total_vendas").desc()).show()

+--------+--------+-----+------------+------------+---------+---------+
| produto|vendedor|total|total_vendas|ticket_medio|max_venda|min_venda|
+--------+--------+-----+------------+------------+---------+---------+
|Notebook|   Carla|    1|      3400.0|      3400.0|   3400.0|   3400.0|
|Notebook|   Bruno|    1|      3300.0|      3300.0|   3300.0|   3300.0|
|Notebook|  Elaine|    1|      3250.0|      3250.0|   3250.0|   3250.0|
|Notebook|     Ana|    1|      3200.0|      3200.0|   3200.0|   3200.0|
|Notebook|   Diego|    1|      3100.0|      3100.0|   3100.0|   3100.0|
| Monitor|   Diego|    1|       980.0|       980.0|    980.0|    980.0|
| Monitor|     Ana|    1|       970.0|       970.0|    970.0|    970.0|
| Monitor|   Bruno|    1|       950.0|       950.0|    950.0|    950.0|
| Teclado|   Bruno|    1|       220.0|       220.0|    220.0|    220.0|
| Teclado|  Elaine|    1|       210.0|       210.0|    210.0|    210.0|
|  Webcam|   Carla|    1|       150.0|       150.0|    150.0|   

In [0]:
## filtro depois de agregar

df.groupBy("produto").agg(
            F.count("*").alias("total"),
            F.sum("valor").alias("total_vendas"),
            F.avg("valor").alias("ticket_medio"),
            F.max("valor").alias("max_venda")
).filter(F.col("ticket_medio")<500).show()

+-------+-----+------------+------------+---------+
|produto|total|total_vendas|ticket_medio|max_venda|
+-------+-----+------------+------------+---------+
| Webcam|    2|       295.0|       147.5|    150.0|
|  Mouse|    3|       237.0|        79.0|     82.0|
|Teclado|    2|       430.0|       215.0|    220.0|
+-------+-----+------------+------------+---------+



In [0]:
## filtro depois de agregar

df.filter(F.col("regiao")=="Sul").groupBy("produto").agg(
            F.count("*").alias("total"),
            F.sum("valor").alias("total_vendas"),
            F.avg("valor").alias("ticket_medio"),
            F.max("valor").alias("max_venda")
).filter(F.col("ticket_medio")<500).show()

+-------+-----+------------+------------+---------+
|produto|total|total_vendas|ticket_medio|max_venda|
+-------+-----+------------+------------+---------+
|Teclado|    1|       220.0|       220.0|    220.0|
+-------+-----+------------+------------+---------+



In [0]:
df.groupBy('produto').sum('valor').show()

+--------+----------+
| produto|sum(valor)|
+--------+----------+
|  Webcam|     295.0|
|   Mouse|     237.0|
| Monitor|    2900.0|
| Teclado|     430.0|
|Notebook|   16250.0|
+--------+----------+



In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy('produto')

df.select('id','produto','valor',
          F.sum('valor').over(w).alias("total_produto"),
          (F.col('valor')/F.sum('valor').over(w)).alias("percentual_produto")
          ).show()

+---+--------+------+-------------+-------------------+
| id| produto| valor|total_produto| percentual_produto|
+---+--------+------+-------------+-------------------+
|  3| Monitor| 950.0|       2900.0| 0.3275862068965517|
|  7| Monitor| 980.0|       2900.0|0.33793103448275863|
| 11| Monitor| 970.0|       2900.0|0.33448275862068966|
|  2|   Mouse|  80.0|        237.0|0.33755274261603374|
|  9|   Mouse|  75.0|        237.0|0.31645569620253167|
| 13|   Mouse|  82.0|        237.0| 0.3459915611814346|
|  1|Notebook|3200.0|      16250.0|0.19692307692307692|
|  5|Notebook|3400.0|      16250.0|0.20923076923076922|
|  8|Notebook|3100.0|      16250.0|0.19076923076923077|
| 12|Notebook|3300.0|      16250.0|0.20307692307692307|
| 15|Notebook|3250.0|      16250.0|                0.2|
|  4| Teclado| 220.0|        430.0| 0.5116279069767442|
| 10| Teclado| 210.0|        430.0| 0.4883720930232558|
|  6|  Webcam| 150.0|        295.0| 0.5084745762711864|
| 14|  Webcam| 145.0|        295.0| 0.4915254237

In [0]:
w = Window.partitionBy('produto').orderBy("data_venda","id")

df.select('id','data_venda','produto','valor',
          F.sum('valor').over(w).alias("acumulado"),
          ).show()

+---+----------+--------+------+---------+
| id|data_venda| produto| valor|acumulado|
+---+----------+--------+------+---------+
|  3|2026-01-10| Monitor| 950.0|    950.0|
|  7|2026-03-03| Monitor| 980.0|   1930.0|
| 11|2026-04-02| Monitor| 970.0|   2900.0|
|  2|2026-01-07|   Mouse|  80.0|     80.0|
|  9|2026-03-15|   Mouse|  75.0|    155.0|
| 13|2026-04-12|   Mouse|  82.0|    237.0|
|  1|2026-01-05|Notebook|3200.0|   3200.0|
|  5|2026-02-11|Notebook|3400.0|   6600.0|
|  8|2026-03-09|Notebook|3100.0|   9700.0|
| 12|2026-04-08|Notebook|3300.0|  13000.0|
| 15|2026-05-06|Notebook|3250.0|  16250.0|
|  4|2026-02-02| Teclado| 220.0|    220.0|
| 10|2026-03-20| Teclado| 210.0|    430.0|
|  6|2026-02-14|  Webcam| 150.0|    150.0|
| 14|2026-05-01|  Webcam| 145.0|    295.0|
+---+----------+--------+------+---------+



In [0]:
w = Window.partitionBy('produto').orderBy(F.col('valor').desc())

df.select('id','data_venda','produto','valor',
          F.row_number().over(w).alias('numeração'),
          ).show()

+---+----------+--------+------+---------+
| id|data_venda| produto| valor|numeração|
+---+----------+--------+------+---------+
|  7|2026-03-03| Monitor| 980.0|        1|
| 11|2026-04-02| Monitor| 970.0|        2|
|  3|2026-01-10| Monitor| 950.0|        3|
| 13|2026-04-12|   Mouse|  82.0|        1|
|  2|2026-01-07|   Mouse|  80.0|        2|
|  9|2026-03-15|   Mouse|  75.0|        3|
|  5|2026-02-11|Notebook|3400.0|        1|
| 12|2026-04-08|Notebook|3300.0|        2|
| 15|2026-05-06|Notebook|3250.0|        3|
|  1|2026-01-05|Notebook|3200.0|        4|
|  8|2026-03-09|Notebook|3100.0|        5|
|  4|2026-02-02| Teclado| 220.0|        1|
| 10|2026-03-20| Teclado| 210.0|        2|
|  6|2026-02-14|  Webcam| 150.0|        1|
| 14|2026-05-01|  Webcam| 145.0|        2|
+---+----------+--------+------+---------+



In [0]:
# Dados de exemplo COM EMPATES de propósito (a Ana e a Eva não empatam, mas Bruno e Carla sim, e Eva e Fabio também)
dados = [
    ("Ana",   "Salvador", 100),
    ("Bruno", "Salvador", 200),
    ("Carla", "Salvador", 200),
    ("Diego", "Salvador", 300),
    ("Eva",   "Recife",   150),
    ("Fabio", "Recife",   150),
    ("Gil",   "Recife",   50),
]
df = spark.createDataFrame(dados, "vendedor STRING, cidade STRING, vendas INT")



In [0]:
# Ranking dentro de cada cidade, do maior para o menor
w = Window.partitionBy("cidade").orderBy(F.col("vendas").desc())

# ---------- 1. AS TRÊS FUNÇÕES LADO A LADO ----------
df.select(
    "cidade", "vendedor", "vendas",
    F.row_number().over(w).alias("row_number"),    # nunca repete: empatados recebem números diferentes
    F.rank().over(w).alias("rank"),                # empatados repetem o número e a próxima posição é PULADA
    F.dense_rank().over(w).alias("dense_rank")     # empatados repetem o número e a próxima posição NÃO é pulada
).orderBy("cidade", "row_number").show()

+--------+--------+------+----------+----+----------+
|  cidade|vendedor|vendas|row_number|rank|dense_rank|
+--------+--------+------+----------+----+----------+
|  Recife|     Eva|   150|         1|   1|         1|
|  Recife|   Fabio|   150|         2|   1|         1|
|  Recife|     Gil|    50|         3|   3|         2|
|Salvador|   Diego|   300|         1|   1|         1|
|Salvador|   Bruno|   200|         2|   2|         2|
|Salvador|   Carla|   200|         3|   2|         2|
|Salvador|     Ana|   100|         4|   4|         3|
+--------+--------+------+----------+----+----------+



In [0]:
from pyspark.sql import functions as F

df = spark.read.parquet(
    '/Volumes/blueprint_spark/bronze/bronze/vendas.parquet',
    mergeSchema=True,
    )

display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df = spark.read.parquet(
    '/Volumes/blueprint_spark/bronze/bronze/vendas.parquet',
    mergeSchema=True,
)

df = df.withColumn("data_venda", F.to_date("data_venda"))

display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
w = Window.partitionBy('vendedor').orderBy('data_venda','id')

df.select(
    "vendedor","data_venda","produto","valor",
    F.lag("valor").over(w).alias("venda_anterior"),
    F.lead("valor").over(w).alias("proxima_venda")
).orderBy('vendedor','data_venda').show()

+--------+----------+--------+------+--------------+-------------+
|vendedor|data_venda| produto| valor|venda_anterior|proxima_venda|
+--------+----------+--------+------+--------------+-------------+
|     Ana|2026-01-05|Notebook|3200.0|          NULL|         80.0|
|     Ana|2026-01-07|   Mouse|  80.0|        3200.0|        970.0|
|     Ana|2026-04-02| Monitor| 970.0|          80.0|         NULL|
|   Bruno|2026-01-10| Monitor| 950.0|          NULL|        220.0|
|   Bruno|2026-02-02| Teclado| 220.0|         950.0|       3300.0|
|   Bruno|2026-04-08|Notebook|3300.0|         220.0|         NULL|
|   Carla|2026-02-11|Notebook|3400.0|          NULL|        150.0|
|   Carla|2026-02-14|  Webcam| 150.0|        3400.0|         82.0|
|   Carla|2026-04-12|   Mouse|  82.0|         150.0|         NULL|
|   Diego|2026-03-03| Monitor| 980.0|          NULL|       3100.0|
|   Diego|2026-03-09|Notebook|3100.0|         980.0|        145.0|
|   Diego|2026-05-01|  Webcam| 145.0|        3100.0|         N